# dbt

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/dbt/dbt.ipynb)

dbt turns SQL `SELECT` statements into tested, documented models that build in dependency order inside your warehouse. This notebook covers models and materializations, incremental strategies, tests, contracts and unit tests, source freshness and slim CI. The runnable cells use DuckDB to show what each pattern does to the data.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {'duckdb': 'duckdb>=1.0', 'dbt': 'dbt-core>=1.9,<2', 'dbt.adapters.duckdb': 'dbt-duckdb>=1.9,<2'}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

## What is dbt?

dbt (data build tool) is a transformation framework: you write transformations as SQL (with Jinja), and dbt builds a DAG of models, runs them in dependency order, tests them, and documents them. It pushes compute into the warehouse (Snowflake, BigQuery, Redshift, Databricks, Postgres, Trino) instead of moving data into a separate processing cluster.

```text
raw sources -> staging (clean) -> intermediate (join/logic) -> marts (facts, dims, metrics)
                    |                     |                        |
                 tests/docs          ref() lineage          exposures to BI
```

dbt is not an orchestrator and not a scheduler. Airflow (or dbt Cloud jobs, Dagster, Prefect) decides *when* dbt runs; dbt decides *what* SQL runs and in what order.

In YAML the generic tests (`unique`, `not_null`, `relationships`, `accepted_values`) sit under `data_tests:`. dbt 1.8 renamed the key from `tests:` to separate them from the new `unit_tests:`; older projects still use `tests:`, which keeps working.

### Where you'll use it

- **SQL transformations under version control:** a warehouse team keeps every model in Git, tested and documented, instead of a folder of ad-hoc scripts.

## How is dbt different from Airflow?

Airflow schedules and coordinates work across systems (extract, dbt, Spark, ML, alerts). dbt owns the SQL transformation graph, its tests, and its documentation. In practice Airflow triggers dbt as one or a few steps inside a larger pipeline: `ingest -> dbt build -> validate -> publish`.

```text
Airflow DAG: extract_orders >> dbt_build(tag:daily) >> publish_marts >> notify
dbt inside:  stg_orders -> int_orders_enriched -> fct_orders -> dim_customer
```

The interview point: dbt has its own DAG, but it cannot wait for a partner file, retry only a failed non-SQL step, or coordinate a Spark job - that is Airflow's job.

### Where you'll use it

- **Division of work:** dbt builds the tables from SQL; Airflow decides when dbt runs and what happens if it fails.

## What is a dbt model?

A model is a `select` statement in a `.sql` file under `models/`. dbt wraps it in the DDL/DML needed to materialize a relation, and resolves dependencies from `ref()` calls.

```sql
-- models/staging/stg_orders.sql
with source as (
    select * from {{ source('postgres', 'orders') }}
),

renamed as (
    select
        id                              as order_id,
        customer_id,
        cast(created_at as timestamp)   as created_at,
        (amount_cents / 100.0)::numeric(18, 2) as amount,
        lower(status)                   as status
    from source
    where _deleted is not true
)

select * from renamed
```

Everything in a model should be deterministic and set-based. If you find yourself writing procedural loops, the logic usually belongs in SQL windows or in an upstream job.

### Where you'll use it

- **Business tables:** a `fct_orders` model is a SELECT that the team publishes as a table for BI tools.

## What does ref do and why is it important?

`ref('model_name')` is the only supported way to refer to another model. It lets dbt build the execution DAG, resolve the right database/schema per environment (dev vs prod), and render lineage in `dbt docs`. Directly hard-coding `analytics.fct_orders` breaks all three.

```sql
-- models/marts/fct_orders.sql
select
    o.order_id,
    o.customer_id,
    o.amount,
    c.segment
from {{ ref('stg_orders') }} as o
left join {{ ref('dim_customer') }} as c
    on o.customer_id = c.customer_id
```

`source('system', 'table')` is the same idea for raw tables you do not build: it gives you freshness checks, lineage from ingestion tools, and a single place to change a schema name.

### Where you'll use it

- **Lineage and order:** `ref('stg_orders')` makes dbt build staging before marts and draw the dependency graph used for impact analysis.

## How do you structure a dbt project?

Separate raw access, cleaning, reusable business logic, and consumer-facing marts. Keep each layer small enough to test and name consistently.

```text
dbt/
  dbt_project.yml
  packages.yml
  models/
    staging/
      _sources.yml
      stg_orders.sql
      stg_customers.sql
    intermediate/
      int_orders_enriched.sql
    marts/
      finance/fct_orders.sql
      core/dim_customer.sql
      core/_models.yml
  snapshots/dim_customer_snapshot.sql
  tests/assert_revenue_non_negative.sql
  macros/cents_to_dollars.sql
```

```yaml
# dbt_project.yml - materialization defaults per folder
name: acme_analytics
version: "1.0.0"
profile: acme
model-paths: ["models"]
```

Naming rules that scale: `stg_` for staging, `int_` for intermediate, `fct_`/`dim_` for marts. One model, one grain, stated in the model description.

### Where you'll use it

- **Team ownership:** folders per domain (sales, finance) let each team own its models and review rules.

## What are the typical dbt layers (staging, intermediate, marts)?

+ **Staging** - rename, cast, deduplicate, and filter raw sources. One staging model per source table, no joins.
+ **Intermediate** - reusable joins and business logic that more than one mart needs. Often ephemeral (CTE-only, no relation).
+ **Marts** - facts, dimensions, and metrics-ready tables that BI and data scientists query. These get the strongest tests and documentation.

This separation stops each dashboard model from re-implementing the same casts and joins, and it makes impact analysis possible when a source changes.

### Where you'll use it

- **Warehouse layering:** staging cleans raw source tables, intermediate joins them, and marts serve analysts. A change in a source touches only staging.

## What is a materialization in dbt?

A materialization is the strategy dbt uses to build a model in the warehouse:

| Materialization | What dbt creates | Use when |
|---|---|---|
| `view` | a database view | cheap transformations, fast iteration, freshness on read |
| `table` | a full rebuild each run | moderate size, expensive joins, stable data |
| `incremental` | insert/merge/append of new rows | large facts, event tables, late-arriving data |
| `ephemeral` | a CTE inlined into downstream models | shared logic without a relation |
| `materialized_view` | warehouse-native MV (where supported) | pre-aggregated, auto-refreshing results |

```sql
{{ config(materialized='table', tags=['finance']) }}

select * from {{ ref('int_orders_enriched') }}
```

### Where you'll use it

- **Cost trade-offs:** a view is free to store but slow to query; a table is costly to build but fast to read for a heavy dashboard.

## When would you use view vs table materializations?

Views cost storage nothing and always reflect the latest data, but every downstream query pays the compute. Tables cost storage and build time but make repeated BI queries cheap and predictable.

Rules of thumb:

+ Staging and light intermediate models: `view` or `ephemeral`.
+ Marts queried by many dashboards: `table`.
+ Very large fact tables: `incremental`.
+ Pre-aggregated metrics queried constantly: `table` refreshed on schedule, or a warehouse materialized view.

The failure mode to avoid is a chain of views on views on views, where a single dashboard query re-scans the raw layer.

### Where you'll use it

- **Dashboard-driven choice:** a model used by one weekly report is a view; a model that powers a live dashboard is a table.

## What is an incremental model and what problems does it solve?

An incremental model processes only new or changed rows instead of rebuilding the whole table. On a billion-row fact table that turns a two-hour rebuild into a two-minute merge.

```sql
-- models/marts/fct_orders.sql
{{
  config(
    materialized='incremental',
    unique_key='order_id',
    incremental_strategy='merge',
    partition_by={'field': 'order_date', 'data_type': 'date'},
    cluster_by=['customer_id']
  )
}}

select
    order_id,
    customer_id,
    order_date,
    amount,
    status,
    updated_at
from {{ ref('int_orders_enriched') }}

{% if is_incremental() %}
    -- only rows changed since the last successful load
    where updated_at > (select coalesce(max(updated_at), timestamp '1900-01-01') from {{ this }})
{% endif %}
```

The `{% if is_incremental() %}` block only applies on incremental runs, so the same model works for the first full build and later merges.

### Where you'll use it

- **Large fact tables:** a model that processes only new orders each night, instead of rebuilding a billion-row table.

## How do you design a reliable unique_key for incremental models?

The `unique_key` must be unique in the target table and stable across source changes - usually a business key (`order_id`) or a deterministic surrogate key built from stable columns (`dbt_utils.generate_surrogate_key`).

Additional rules:

+ Keep a monotonic `updated_at` (or CDC sequence) so you can filter "changed since last run".
+ Use `incremental_strategy='merge'` for updates; `append` only for immutable events; `delete+insert` for warehouses without merge.
+ Add a small lookback window (for example `updated_at > max(updated_at) - interval '3 days'`) to absorb late-arriving rows and clock skew.
+ Handle deletes explicitly: CDC with a `_deleted` flag or the `hard_deletes` snapshot config (dbt 1.9+), otherwise deleted rows live forever.
+ Make the key unique in the source too; if not, deduplicate in the staging model with a window function.

```sql
-- deduplicate the source before merging
select *
from (
    select
        *,
        row_number() over (partition by order_id order by updated_at desc) as rn
    from {{ source('postgres', 'orders') }}
) where rn = 1
```

### Where you'll use it

- **Upserts in the warehouse:** a `unique_key` of `order_id` updates changed orders instead of inserting duplicates.

## How do you handle backfills and late-arriving data in dbt?

Parameterize the window and let orchestration pass it. Never hand-edit SQL to backfill.

```sql
-- use a var with a sensible default so production and backfills share one model
{% set start_date = var('start_date', '1900-01-01') %}
{% set end_date = var('end_date', '2999-12-31') %}

select *
from {{ ref('int_orders_enriched') }}
where order_date between date '{{ start_date }}' and date '{{ end_date }}'
```

```bash
# reprocess one month
dbt build --select fct_orders --vars '{"start_date": "2026-09-01", "end_date": "2026-09-30"}'
```

For late-arriving data, combine a lookback window with idempotent merges so reprocessing a day is safe. If the model cannot be made idempotent, rebuild it fully for the affected partitions instead of appending.

### Where you'll use it

- **Late corrections:** a lookback window reprocesses the last three days, so an order that arrives late is still counted.

## What are dbt tests and what types exist?

+ **Generic tests** - reusable, declared in YAML: `unique`, `not_null`, `accepted_values`, `relationships`, plus packages such as `dbt_utils` and `dbt_expectations`.
+ **Singular tests** - a SQL file under `tests/` that returns offending rows; the test passes when it returns zero rows.
+ **Unit tests** - dbt 1.8+ fixtures that verify transformation logic on fixed inputs without touching the warehouse.

```sql
-- tests/assert_revenue_non_negative.sql
select *
from {{ ref('fct_orders') }}
where amount < 0
```

```sql
-- tests/generic/test_not_future.sql : a custom generic test
{% test not_future(model, column_name) %}
select *
from {{ model }}
where {{ column_name }} > current_timestamp
{% endtest %}
```

Tests run as SQL in the warehouse, so they cost compute: put the expensive ones on critical marts, not on every staging view.

### Where you'll use it

- **Data contracts in CI:** `not_null` and `unique` tests run on every pull request, so a broken key fails review, not the dashboard.

## What dbt tests should you add to critical models?

Protect the model's contract, not random columns: uniqueness and nullability of keys, accepted values, referential integrity to dimensions, non-negative amounts, and one current row per key in SCD2 tables.

```yaml
# models/marts/core/_models.yml
models:
  - name: fct_orders
    description: One row per order, at the order grain.
    config:
      contract:
        enforced: true
    columns:
      - name: order_id
        data_tests: [unique, not_null]
      - name: customer_id
        data_tests:
          - not_null
          - relationships:
              to: ref('dim_customer')
              field: customer_id
      - name: status
        data_tests:
          - accepted_values:
              values: ["placed", "shipped", "delivered", "cancelled"]
      - name: amount
        data_tests:
          - dbt_utils.expression_is_true:
              expression: ">= 0"
```

Also check **freshness** on sources and **volume** (row counts vs trailing average) so a silently empty load fails the run instead of the dashboard.

### Where you'll use it

- **Revenue tables:** uniqueness of the order key, non-null amounts and a relationship test to customers protect the numbers finance reads.

## What are sources in dbt and how do you test them?

Sources declare raw tables so dbt can track freshness and lineage from ingestion to BI.

```yaml
# models/staging/_sources.yml
sources:
  - name: postgres
    database: raw
    schema: public
    loaded_at_field: _ingested_at
    freshness:
      warn_after: {count: 6, period: hour}
      error_after: {count: 24, period: hour}
    tables:
      - name: orders
        columns:
          - name: id
            data_tests: [unique, not_null]
      - name: customers
```

```bash
dbt source freshness          # fails when a source is stale past error_after
```

Freshness is the cheapest early warning you can buy: it catches broken ingestion before transformations run on stale or partial data.

### Where you'll use it

- **Raw-data freshness:** a source freshness check alerts when the vendor's table has not been updated for 24 hours.

## What are snapshots in dbt and when would you use them?

Snapshots capture the history of a mutable table so you can answer "what did this row look like on date X". They implement SCD2 with `dbt_valid_from` / `dbt_valid_to`.

```sql
-- snapshots/dim_customer_snapshot.sql
{% snapshot dim_customer_snapshot %}
{{
  config(
    target_schema='snapshots',
    unique_key='customer_id',
    strategy='timestamp',
    updated_at='updated_at'
  )
}}
select * from {{ source('postgres', 'customers') }}
{% endsnapshot %}
```

Use `strategy='check'` when there is no reliable `updated_at` (dbt compares a list of columns). Snapshots are append-only history - they grow forever, so plan retention and never "fix" history by editing them.

### Where you'll use it

- **Tracking history:** a snapshot of the customer table records each change, which answers questions about past attributes.

## What are macros in dbt and when should you use them?

Macros are Jinja functions that generate SQL. Use them for repeated column logic, standard filters, and cross-database compatibility - not for hiding business rules that reviewers must approve.

```sql
-- macros/cents_to_dollars.sql
{% macro cents_to_dollars(column_name, scale=2) %}
    ({{ column_name }} / 100.0)::numeric(16, {{ scale }})
{% endmacro %}
```

```sql
select
    order_id,
    {{ cents_to_dollars('amount_cents') }} as amount
from {{ ref('stg_orders') }}
```

Macros can be tested indirectly through the models that use them, and `dbt run-operation` lets you call one directly for operational tasks.

### Where you'll use it

- **Repeated logic:** a `cents_to_dollars` macro is used in twenty models, so a rounding change is made once.

## How do you document and expose dbt models?

Documentation lives with the model and is rendered by `dbt docs generate`. Exposures declare downstream consumers (dashboards, ML models) so impact analysis reaches BI.

```yaml
exposures:
  - name: revenue_dashboard
    type: dashboard
    url: https://bi.acme.com/dashboards/revenue
    owner:
      name: Finance Analytics
      email: finance-data@acme.com
    depends_on:
      - ref('fct_orders')
      - ref('dim_customer')
```

Add `description` and `meta` fields to models, and enforce `contract: enforced: true` on marts that other teams consume so a column rename fails fast instead of breaking a dashboard silently.

### Where you'll use it

- **Self-service analytics:** descriptions and exposures list which dashboards depend on a model, so analysts know what a change will affect.

## How do you orchestrate dbt with Airflow?

Run dbt as a step inside a larger DAG. Airflow handles waiting for data, retries, and non-SQL work; dbt handles the SQL graph.

Airflow 3 (Airflow 2 imports `BashOperator` from `airflow.operators.bash` and `task` from `airflow.decorators`):

```python
from airflow.providers.standard.operators.bash import BashOperator
from airflow.sdk import DAG, task

with DAG(dag_id="daily_revenue", schedule="0 3 * * *", start_date=...) as dag:

    wait_for_orders = S3KeySensor(
        task_id="wait_for_orders",
        bucket_name="acme-raw",
        bucket_key="orders/ds={{ ds }}/_SUCCESS",
        mode="reschedule",
    )

    dbt_build = BashOperator(
        task_id="dbt_build",
        bash_command="cd /opt/dbt/acme && dbt build --select tag:daily --target prod",
    )

    @task
    def publish() -> None:
        notify_bi_refresh()

    wait_for_orders >> dbt_build >> publish()
```

Alternatives worth naming: dbt Cloud jobs triggered by `DbtCloudRunJobOperator`, or Cosmos, which renders each dbt model as an Airflow task so retries and logs are per model.

### Where you'll use it

- **Scheduled transformations:** an Airflow task runs `dbt build` after the load and fails the DAG when a test fails.

## What is dbt state selection (slim CI) and why is it useful?

State selection compares the current project against a previous manifest and selects only changed models plus their downstream dependents. In CI this avoids rebuilding the whole warehouse while still validating the affected part of the DAG.

```bash
# CI: run only what changed, deferring unmodified refs to the production manifest
dbt build \
  --select state:modified+ \
  --defer \
  --state ./prod-manifest \
  --target ci
```

`--defer` makes unmodified `ref()`s resolve to production, so the CI schema only contains the models under test. Generate the manifest in production on every deploy (`dbt build --select ...` then copy `target/manifest.json`) and store it as an artifact.

### Where you'll use it

- **Fast pull-request checks:** slim CI builds only the models changed in the pull request and their children, so the check takes minutes, not hours.

## How do you approach CI/CD for dbt projects?

A practical pipeline:

+ **CI on every PR** - `dbt deps`, `dbt build --select state:modified+ --defer --state ./prod-manifest --target ci`, then run the full test suite on the changed selection.
+ **CD on merge** - deploy the project, then run `dbt build` for the changed selection in production (or the full graph nightly).
+ **Guardrails** - code review for marts, enforced contracts, source freshness alerts, and a rollback plan (revert the PR, then rebuild affected models).
+ **Observability** - store `run_results.json` and `manifest.json` artifacts, alert on failures and on tests that start failing repeatedly.

```yaml
# .github/workflows/dbt.yml (sketch)
steps:
  - run: dbt deps
  - run: dbt build --select state:modified+ --defer --state prod-manifest --target ci
  - run: dbt test --select state:modified+
```

### Where you'll use it

- **Promotion between environments:** a pull request builds into a temporary schema, and merge deploys to production with the same code.

## How do you tune dbt performance?

+ **Reduce data scanned** - partition and cluster large facts, filter early in staging, avoid `select *` across layers.
+ **Choose the right materialization** - the biggest lever; a table instead of a view chain can cut warehouse cost dramatically.
+ **Increase threads** - `dbt run --threads 8` (bounded by warehouse concurrency and cost).
+ **Make incrementals correct and narrow** - a tight `updated_at` filter plus a small lookback beats a full rebuild.
+ **Precompute heavy joins** - materialize the intermediate model once instead of joining raw tables in every mart.
+ **Watch full-refresh cost** - `dbt build --full-refresh` on a huge incremental is an incident if it is not planned.

```bash
dbt run --select fct_orders --threads 8
dbt build --select tag:finance --exclude tag:experimental
dbt run --full-refresh --select fct_orders     # deliberate, not routine
```

### Where you'll use it

- **Slow nightly builds:** a team finds the three heaviest models and changes them to incremental, which cuts the build by hours.

## What are common dbt anti-patterns?

+ One giant model with hundreds of CTEs that nobody can test or review.
+ Views on views on views, so every dashboard query re-scans raw data.
+ Hard-coded schema/table names instead of `ref()` and `source()`.
+ Incremental models without a `unique_key`, causing duplicate rows on retry.
+ Tests on trivial columns while key relationships and freshness go unchecked.
+ Business logic in macros that reviewers cannot see in the model SQL.
+ `dbt run` in production without tests (`dbt build` runs both).
+ Snapshots used as a substitute for CDC and then never pruned.
+ No manifest artifact, so CI cannot use state selection.
+ Treating dbt as a scheduler; complex non-SQL orchestration does not belong in a dbt project.

### Where you'll use it

- **Review checklist:** business logic in a staging model, a mart that reads a raw table directly, and no tests on keys are the usual findings in a dbt review.

## Real world use cases for dbt

### 1. E-commerce revenue marts on Snowflake

Raw orders, payments, and refunds land from CDC. Staging models clean them; intermediate models join and allocate refunds; marts produce `fct_orders`, `fct_payments`, and `dim_customer` used by finance dashboards.

```sql
-- fct_orders: one row per order, incremental merge, partitioned by date
{{ config(materialized='incremental', unique_key='order_id', incremental_strategy='merge',
          partition_by={'field': 'order_date', 'data_type': 'date'}) }}

select o.order_id, o.customer_id, o.order_date, o.amount - coalesce(r.refunded, 0) as net_amount
from {{ ref('stg_orders') }} o
left join {{ ref('int_refunds_by_order') }} r using (order_id)
{% if is_incremental() %}
where o.updated_at > (select coalesce(max(updated_at), '1900-01-01') from {{ this }})
{% endif %}
```

Why dbt: one tested, documented definition of revenue, rebuilt incrementally instead of re-derived in every dashboard.

### 2. Fintech regulatory reporting with SCD2 history

A licensed lender must reproduce "the customer segment as of the reporting date". A snapshot keeps SCD2 history for `customers`, and marts join each transaction to the row valid on its booking date.

```sql
select
    t.transaction_id,
    t.amount,
    s.customer_segment,
    s.dbt_valid_from,
    s.dbt_valid_to
from {{ ref('fct_transactions') }} t
join {{ ref('dim_customer_snapshot') }} s
  on t.customer_id = s.customer_id
 and t.booked_at >= s.dbt_valid_from
 and t.booked_at <  coalesce(s.dbt_valid_to, timestamp '9999-12-31')
```

Why dbt: snapshots make point-in-time correctness a modelling concern rather than an audit scramble.

### 3. Multi-tenant SaaS metrics

Each tenant's usage events are transformed into tenant-scoped marts, with tests asserting that tenant ids never leak across accounts.

```sql
select tenant_id, date_trunc('day', event_at) as ds, count(*) as events
from {{ ref('stg_usage_events') }}
group by 1, 2
```

```yaml
models:
  - name: fct_usage_daily
    columns:
      - name: tenant_id
        data_tests:
          - not_null
          - relationships: {to: ref('dim_tenant'), field: tenant_id}
```

Why dbt: consistent tenant-level metric definitions and contract tests, which is exactly what a multi-tenant data product needs.

### 4. Migrating stored procedures to governed SQL

A team replaces a 4,000-line stored procedure with staging, intermediate, and mart models. Each step gets tests, and the final numbers are reconciled against the legacy procedure before cutover.

```bash
dbt build --select +fct_billing_statement     # build the model and all its ancestors
dbt test --select fct_billing_statement
```

Why dbt: the migration becomes reviewable and testable, and the lineage makes the replacement plan obvious.

### 5. Feature tables for machine learning

Data scientists need the same feature definitions in training and serving. dbt builds `fct_customer_features` with a documented grain and freshness test; the training pipeline reads the table, and Airflow triggers retraining when the model is rebuilt.

Why dbt: training/serving skew drops when both sides read one versioned, tested definition instead of two copies of SQL.

### Try it - a tiny dbt on DuckDB

The real `dbt` CLI needs a project and adapter, but its core loop is small: resolve `ref()` to a relation in the target schema, order models by their parents, and run `CREATE VIEW/TABLE AS SELECT`. DuckDB stands in for the warehouse.

In [ ]:
# A tiny dbt: models are SELECTs with {{ ref() }}, built in DAG order into DuckDB.
import re

import duckdb

con = duckdb.connect()
con.execute("""
    CREATE SCHEMA raw;
    CREATE TABLE raw.orders AS SELECT * FROM (VALUES
        (1, 'C1', TIMESTAMP '2026-10-01 09:00', 4250, 'PLACED',    FALSE),
        (2, 'C2', TIMESTAMP '2026-10-01 10:30', 1700, 'Shipped',   FALSE),
        (3, 'C1', TIMESTAMP '2026-10-02 08:15',  999, 'delivered', FALSE),
        (4, 'C3', TIMESTAMP '2026-10-02 11:00', 5000, 'cancelled', TRUE)
    ) AS t(id, customer_id, created_at, amount_cents, status, _deleted);
    CREATE TABLE raw.customers AS SELECT * FROM (VALUES
        ('C1', 'enterprise'), ('C2', 'smb'), ('C3', 'smb')
    ) AS t(customer_id, segment);
""")

MODELS = {  # name -> (materialization, SQL); the same shape as files under models/
    "stg_orders": ("view", """
        select id as order_id, customer_id, created_at,
               cast(amount_cents / 100.0 as decimal(18, 2)) as amount,
               lower(status) as status
        from raw.orders
        where _deleted is not true"""),
    "stg_customers": ("view", "select customer_id, segment from raw.customers"),
    "dim_customer": ("table", "select * from {{ ref('stg_customers') }}"),
    "fct_orders": ("table", """
        select o.order_id, o.customer_id, o.amount, o.status, c.segment
        from {{ ref('stg_orders') }} as o
        left join {{ ref('dim_customer') }} as c using (customer_id)"""),
}
REF = re.compile(r"\{\{\s*ref\('([a-z_]+)'\)\s*\}\}")


def parents(sql: str) -> list[str]:
    return REF.findall(sql)


def build_order(models: dict) -> list[str]:
    order, remaining = [], dict(models)
    while remaining:
        ready = sorted(n for n, (_, sql) in remaining.items() if all(p in order for p in parents(sql)))
        if not ready:
            raise ValueError(f"cycle among {sorted(remaining)}")
        order += ready
        for name in ready:
            del remaining[name]
    return order


def run(models: dict, schema: str = "analytics") -> list[str]:
    con.execute(f"CREATE SCHEMA IF NOT EXISTS {schema}")
    order = build_order(models)
    for name in order:
        materialization, sql = models[name]
        rendered = REF.sub(lambda m: f"{schema}.{m.group(1)}", sql)   # ref() resolves per target
        con.execute(f"CREATE OR REPLACE {materialization} {schema}.{name} AS {rendered}")
    return order


order = run(MODELS)
print("built:", " -> ".join(order))
print(con.sql("select * from analytics.fct_orders order by order_id"))
assert order.index("dim_customer") < order.index("fct_orders")
assert con.execute("select count(*) from analytics.fct_orders").fetchall()[0][0] == 3   # deleted row filtered

### Try it - generic tests as SQL

`unique`, `not_null`, `accepted_values`, `relationships`, and `dbt_utils.expression_is_true` all compile to a query that returns offending rows.

In [ ]:
# dbt tests compile to SELECTs that return failing rows; zero rows means the test passes.
def unique(model, column):
    return f"select {column} from {model} group by {column} having count(*) > 1"


def not_null(model, column):
    return f"select * from {model} where {column} is null"


def accepted_values(model, column, values):
    allowed = ", ".join(f"'{v}'" for v in values)
    return f"select * from {model} where {column} not in ({allowed})"


def relationships(model, column, to, field):
    return (f"select m.* from {model} m left join {to} p on m.{column} = p.{field} "
            f"where m.{column} is not null and p.{field} is null")


def expression_is_true(model, expression):
    return f"select * from {model} where not ({expression})"


TESTS = {
    "unique_order_id": unique("analytics.fct_orders", "order_id"),
    "not_null_order_id": not_null("analytics.fct_orders", "order_id"),
    "accepted_status": accepted_values("analytics.fct_orders", "status",
                                       ["placed", "shipped", "delivered", "cancelled"]),
    "customer_fk": relationships("analytics.fct_orders", "customer_id",
                                 "analytics.dim_customer", "customer_id"),
    "amount_non_negative": expression_is_true("analytics.fct_orders", "amount >= 0"),
}


def dbt_test() -> dict[str, int]:
    return {name: len(con.execute(sql).fetchall()) for name, sql in TESTS.items()}


results = dbt_test()
print("clean run:", results)
assert all(n == 0 for n in results.values())

# A bad load: a duplicate order with an unknown customer and a new status.
con.execute("INSERT INTO raw.orders VALUES (2, 'C9', TIMESTAMP '2026-10-03 07:00', 100, 'lost', FALSE)")
run(MODELS)
failed = {name: n for name, n in dbt_test().items() if n}
print("after bad load:", failed)
assert set(failed) == {"unique_order_id", "accepted_status", "customer_fk"}
con.execute("DELETE FROM raw.orders WHERE customer_id = 'C9'")
run(MODELS)

### Try it - an incremental merge in SQL

This is what `materialized='incremental'` with `unique_key` and `incremental_strategy='merge'` generates, written by hand in DuckDB SQL.

In [ ]:
# An incremental model: merge only rows changed since the last run (with a lookback)
# on unique_key, so reruns are idempotent and stale rows never overwrite newer ones.
con.execute("""
    CREATE TABLE src_orders (order_id INT, amount DECIMAL(10, 2), updated_at TIMESTAMP);
    INSERT INTO src_orders VALUES
        (1, 10.00, TIMESTAMP '2026-10-01 00:00'),
        (2, 20.00, TIMESTAMP '2026-10-01 00:00');
    CREATE TABLE fct_orders_incr (order_id INT PRIMARY KEY, amount DECIMAL(10, 2), updated_at TIMESTAMP);
""")

INCREMENTAL_SQL = """
INSERT INTO fct_orders_incr
SELECT order_id, amount, updated_at FROM (
    SELECT *, row_number() OVER (PARTITION BY order_id ORDER BY updated_at DESC) AS rn
    FROM src_orders
    WHERE updated_at > (SELECT coalesce(max(updated_at), TIMESTAMP '1900-01-01')
                        FROM fct_orders_incr) - INTERVAL 3 DAY      -- is_incremental() + lookback
) WHERE rn = 1                                                       -- dedupe the source first
ON CONFLICT (order_id) DO UPDATE SET amount = excluded.amount, updated_at = excluded.updated_at
WHERE excluded.updated_at > fct_orders_incr.updated_at               -- newer version wins
"""

con.execute(INCREMENTAL_SQL)                  # first run behaves like a full build
con.execute("""INSERT INTO src_orders VALUES
    (2, 25.00, TIMESTAMP '2026-10-02 00:00'),   -- update
    (3, 30.00, TIMESTAMP '2026-10-02 00:00'),   -- insert
    (3, 31.00, TIMESTAMP '2026-10-02 06:00')    -- same key twice in one batch
""")
con.execute(INCREMENTAL_SQL)
first = con.execute("SELECT * FROM fct_orders_incr ORDER BY order_id").fetchall()
con.execute(INCREMENTAL_SQL)                  # a retry changes nothing
assert con.execute("SELECT * FROM fct_orders_incr ORDER BY order_id").fetchall() == first
assert [(r[0], float(r[1])) for r in first] == [(1, 10.0), (2, 25.0), (3, 31.0)]
print(con.sql("SELECT * FROM fct_orders_incr ORDER BY order_id"))

### Try it - Build order from ref()

dbt builds a DAG, not a folder order: a model runs only after every model it `ref()`s. Compute that ordering yourself and you will never be surprised by what runs first.

In [ ]:
# Task: return the order dbt should build these models in, given each model's
#   ref() list.
# Hint: a model is ready when all of its parents are already built; sort the
#   ready set so the output is deterministic, and treat "nothing is ready" as
#   a cycle.
# Expected: staging models first, then dim_customer and the intermediate
#   model, then fct_orders.

MODELS = {
    "stg_orders": [],
    "stg_customers": [],
    "int_orders_enriched": ["stg_orders", "stg_customers"],
    "fct_orders": ["int_orders_enriched"],
    "dim_customer": ["stg_customers"],
}


def build_order(models):
    order = []
    remaining = dict(models)
    while remaining:
        ready = sorted(name for name, deps in remaining.items()
                       if all(dep in order for dep in deps))
        if not ready:
            raise ValueError(f"cycle among {sorted(remaining)}")
        order.extend(ready)
        for name in ready:
            del remaining[name]
    return order


expected = ["stg_customers", "stg_orders", "dim_customer",
            "int_orders_enriched", "fct_orders"]
assert build_order(MODELS) == expected, build_order(MODELS)
print("build order:", " -> ".join(build_order(MODELS)))

try:
    build_order({"a": ["b"], "b": ["a"]})
except ValueError as exc:
    print("cycle detected:", exc)

### Try it - Incremental merge on unique_key

An incremental model trusts `unique_key` for identity and a monotonic column for recency. Apply that merge by hand to see why a stale row must not overwrite a newer one.

In [ ]:
# Task: apply a dbt incremental merge. An incoming row replaces the target row
#   with the same order_id only when its updated_at is newer; otherwise it is
#   inserted.
# Hint: unique_key gives identity and updated_at decides which version wins,
#   so a late-arriving older row is ignored rather than applied.
# Expected: order 2 keeps the newer amount, order 3 is inserted, and the stale
#   order 1 row is dropped.

TARGET = [
    {"order_id": 1, "amount": 10.0, "updated_at": 100},
    {"order_id": 2, "amount": 20.0, "updated_at": 100},
]
INCOMING = [
    {"order_id": 2, "amount": 25.0, "updated_at": 200},   # update
    {"order_id": 3, "amount": 30.0, "updated_at": 150},   # insert
    {"order_id": 1, "amount": 5.0,  "updated_at": 50},    # stale, ignore
]


def merge(target, incoming, key="order_id"):
    current = {row[key]: row for row in target}
    for row in incoming:
        existing = current.get(row[key])
        if existing is None or row["updated_at"] > existing["updated_at"]:
            current[row[key]] = row
    return sorted(current.values(), key=lambda row: row[key])


merged = merge(TARGET, INCOMING)
assert [(r["order_id"], r["amount"]) for r in merged] == [(1, 10.0), (2, 25.0), (3, 30.0)]
assert len({r["order_id"] for r in merged}) == len(merged)   # key stays unique
assert merge(merged, INCOMING) == merged                     # rerun is idempotent
print("merged rows:", merged)

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** What does dbt do that Airflow does not, and the other way around?

**A.** dbt owns the SQL transformation graph: it resolves `ref()` dependencies, runs models in order, tests and documents them. Airflow schedules and coordinates work across systems, so it waits for a partner file, retries a non-SQL step, or triggers a Spark job. In practice Airflow triggers dbt as one step inside a larger pipeline.

**Q2.** Why must models use `ref()` instead of hard-coded table names?

**A.** `ref()` lets dbt build the execution DAG, resolve the right database and schema per environment so dev does not write to prod, and render lineage in `dbt docs`. Hard-coding `analytics.fct_orders` breaks all three at once.

**Q3.** When do you choose an incremental materialization over a table?

**A.** Use incremental for large facts and event tables where a full rebuild costs hours, so only new or changed rows are merged. Keep small or moderately sized models as tables for predictability, and remember that incremental adds key, lookback, and delete-handling obligations.

**Q4.** What makes a good `unique_key`, and what else does an incremental model need?

**A.** The key must be unique in the target table and stable across source changes, usually a business key or a deterministic surrogate key. The model also needs a monotonic `updated_at` or CDC sequence to filter changed rows, a small lookback window for late arrivals, and an explicit plan for deletes.

**Q5.** How do you backfill without editing a model by hand?

**A.** Parameterize the window with a `var` that defaults to the full range and let the orchestrator pass real dates, so production and backfills share one model. Combine that with idempotent merges so reprocessing a day replaces rather than duplicates rows, and rebuild fully if a model cannot be made idempotent.

**Q6.** What is the difference between a generic test and a singular test?

**A.** Generic tests are reusable and declared in YAML, such as `unique`, `not_null`, `accepted_values`, and `relationships`, and can be custom. Singular tests are a SQL file under `tests/` that returns offending rows, and the test passes when it returns zero rows. Both run as SQL in the warehouse and cost compute.

**Q7.** How does state selection make CI cheap on a large dbt project?

**A.** State selection compares the current project against a previous manifest and selects only changed models plus their downstream dependents. Adding `--defer` makes unchanged `ref()`s resolve to production, so the CI schema holds only the models under test and a small PR does not rebuild the whole warehouse.

## Run a dbt project against DuckDB

This runs the actual dbt Core Python runner and dbt-duckdb adapter in a temporary project: seed → model → not-null and unique tests. It needs no warehouse credentials. Expected model rows: customer 1 has revenue 30, customer 2 has 5. The v1 adapter is pinned below 2; this example does not require the separate Fusion engine.

[Adapter source and setup](https://github.com/duckdb/dbt-duckdb)

In [ ]:
import tempfile
from pathlib import Path
import duckdb
from dbt.cli.main import dbtRunner

with tempfile.TemporaryDirectory(prefix='dbt-project-') as directory:
    root=Path(directory)
    (root/'models').mkdir(); (root/'seeds').mkdir()
    (root/'dbt_project.yml').write_text("name: local_handbook\nversion: '1.0'\nconfig-version: 2\nprofile: local_handbook\n")
    (root/'profiles.yml').write_text("local_handbook:\n  target: dev\n  outputs:\n    dev:\n      type: duckdb\n      path: '"+str(root/'warehouse.duckdb')+"'\n      threads: 1\n")
    (root/'seeds/orders.csv').write_text('id,customer_id,amount\n1,1,10\n2,1,20\n3,2,5\n')
    (root/'models/revenue.sql').write_text("select customer_id, sum(amount) as revenue from {{ ref('orders') }} group by customer_id")
    (root/'models/schema.yml').write_text('version: 2\nmodels:\n  - name: revenue\n    columns:\n      - name: customer_id\n        data_tests: [not_null, unique]\n')
    result=dbtRunner().invoke(['build','--project-dir',str(root),'--profiles-dir',str(root)])
    assert result.success, str(result.exception)
    with duckdb.connect(str(root/'warehouse.duckdb')) as connection:
        assert connection.sql('select * from revenue order by customer_id').fetchall()==[(1,30),(2,5)]
    print('dbt seed, model and tests passed')